# 🪪 ID Card Tampering Detection: Quickstart

[![GitHub](https://img.shields.io/badge/GitHub-id--tamper--check-181717?logo=github)](https://github.com/saineshnakra/Identification-Card-Fraud-Detection)

This notebook compares a trusted **reference** image of a document with a **candidate** image, aligns them, computes a
Structural Similarity (SSIM) map with OpenCV and scikit-image, and boxes the regions that were edited.

> Use synthetic or specimen documents only. This is an explainable image-difference demo, not an identity-verification
> or fraud-scoring system.

In [ ]:
%pip install -q git+https://github.com/saineshnakra/Identification-Card-Fraud-Detection.git

## 1. Generate a synthetic specimen and a tampered phone photo

`specimen_pair(capture=True)` draws a fictional *Republic of Utopia* card, edits its date of birth, document number and
portrait, then simulates a handheld photo (rotation, perspective, uneven lighting, sensor noise).

In [ ]:
import matplotlib.pyplot as plt

from id_tamper_check import compare_images, render_report
from id_tamper_check.samples import specimen_pair

reference, candidate = specimen_pair(capture=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for axis, image, title in zip(axes, (reference, candidate), ("Reference", "Candidate photo"), strict=True):
    axis.imshow(image)
    axis.set_title(title)
    axis.axis("off")

## 2. Align, compare and visualize

`align=True` registers the photo onto the reference with ORB features and a RANSAC homography, then refines it to
sub-pixel accuracy with ECC before computing SSIM.

In [ ]:
result = compare_images(reference, candidate, align=True)

print(f"SSIM score: {result.score:.4f}")
print(f"Changed regions: {len(result.regions)}")
for region in result.regions:
    print("  ", region)

plt.figure(figsize=(16, 12))
plt.imshow(render_report(result))
plt.axis("off");

## 3. Machine-readable output

`to_dict()` returns JSON-serializable numbers, including the mean difference inside each region.

In [ ]:
import json

print(json.dumps(result.to_dict(), indent=2))

## 4. Try your own images (optional)

Upload a reference and a candidate of the **same document layout**. Prefer specimens or documents you are authorized to
process. Uncheck alignment (`align=False`) for scans that are already cropped identically.

In [ ]:
try:
    from google.colab import files
except ImportError:
    files = None
    print(
        "Not running in Colab: call compare_documents(Path('reference.png'), Path('candidate.png')) instead."
    )

if files:
    from pathlib import Path

    from id_tamper_check import compare_documents

    print("Upload the REFERENCE image")
    reference_path = Path(next(iter(files.upload())))
    print("Upload the CANDIDATE image")
    candidate_path = Path(next(iter(files.upload())))

    result = compare_documents(reference_path, candidate_path, align=True)
    plt.figure(figsize=(16, 12))
    plt.imshow(render_report(result))
    plt.axis("off")

## How to read the result

- **SSIM score** is 1.0 for identical images. It summarizes how similar the images are; it is *not* a fraud probability.
- **Changed regions** are areas whose local dissimilarity exceeds both Otsu's threshold and the `min_difference` floor
  (default 0.5), merged so that an edited field becomes one box.
- Capture conditions (glare, blur, heavy compression) can lower the score without any tampering, and a high score does
  not prove a document is genuine.

⭐ If this was useful, [star the repository](https://github.com/saineshnakra/Identification-Card-Fraud-Detection).